In [ ]:
from typing_extensions import TypedDict, Annotated
from typing import Dict, List
import os
import operator

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages  #reducer funct, x non sovrascrivere quindi ottenere 
#[
#   HumanMessage("hi"),
#   AIMessage("hello")
# ]
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

from pydantic import BaseModel, Field

from scripts import my_tools

from dotenv import load_dotenv
load_dotenv()

LLM_MODEL = "qwen3"
BASE_URL = "http://localhost:11434"

llm = ChatOllama(
    model=LLM_MODEL, 
    base_url=BASE_URL)

In [ ]:
class AgentState(TypedDict):
    messages: Annotated[list, operator.add]
    retrieved_docs: str
    is_relevant: bool
    rewritten_query: str

In [ ]:
result = my_tools.web_search.invoke({'query':'python tutorial', 'num_results': 3})
print(result)
result = my_tools.retrieve_docs.invoke({'query':'google revenue in 2023', 'k': 2})

In [ ]:
def retrieve_node(state:AgentState):
    print(f"[RETRIEVE NODE] Fetching documents")
    user_question = state['messages'][-1].content
    print(f"[Query] User Query: {user_question}")
    result = my_tools.retrieve_docs.invoke({'query': user_question, 'k': 5})
    print(f"[RETRIEVE NODE] Fetched Relevant Documents")
    with open('debug_logs/crag_retrieved_docs.md', 'w', encoding='utf-8') as f:
        f.write(f"Query: {user_question}\n\n")
        f.write(result)
    return {'retrieved_docs': result}

In [ ]:
class GradeDecision(BaseModel):
    is_relevant:bool = Field(description="True if documents are relevant  to answer the question, False if irrelevant")
    reasoning: str = Field(description="Brief explanation of why documens are relevant or not.")

In [ ]:
def grade_node(state:AgentState):
    llm_structured = llm.with_structured_output(GradeDecision)
    user_question = state['messages'][-1].content
    retrieved_docs = state.get('retrieved_docs', '')
    prompt = f"""You are a document relevance grader.
        TASK: Evaluate if the retrieved documents are relevant to answer the user's question.
        USER QUESTION: {user_question}
        RETRIEVED DOCUMENTS:
        {retrieved_docs}
        GRADING CRITERIA:
        - is_relevant = True: If documents contain information that can answer the question
        - is_relevant = False: If documents are completely irrelevant or off-topic
        Output JSON:
        {{
        "is_relevant": true/false,
        "reasoning": "Brief explanation..."
        }}"""
    response = llm_structured.invoke(prompt)
    print(f"[GRADE] Relevant: {response.is_relevant}\nReasoning: {response.reasoning}")
    return {'is_relevant': response.is_relevant}  #update stato shared

def rewrite_query_node(state:AgentState):
    user_question = state['messages'][-1].content
    prompt = f"""You are a query rewriting expert.
        TASK: Rewrite the user's question to make it more specific and targeted for document retrieval.
        ORIGINAL QUESTION: {user_question}
        INSTRUCTIONS:
        - Make the query more specific with keywords
        - Add relevant financial terms (revenue, profit, earnings, etc.)
        - Include company names, years, quarters if mentioned
        - Keep it concise (one sentence)
        Output ONLY the rewritten query, nothing else.""" 
    response = llm.invoke(prompt)
    rewritten_query = response.content.strip()
    print(f"[REWRITE] Original Query: {user_question}")
    print(f"[REWRITE] New: {rewritten_query}")
    return {'rewritten_query': rewritten_query}

def web_search_node(state:AgentState):
    user_question = state['messages'][-1].content
    rewritten_query = state.get("rewritten_query", user_question)
    result = my_tools.web_search.invoke({'query':rewritten_query})
    with open('debug_logs/crag_retry_websearch_docs.md', 'w', encoding='utf-8') as f:
        f.write(f"Rewritten Query: {rewritten_query}\n\n")
        f.write(result)
    return {'retrieved_docs': result}

def answer_node(state:AgentState):
    # messages: [HumanMessage]
    user_question = state['messages'][-1].content
    retrieved_docs = state.get('retrieved_docs', '')
    prompt = f"""You are an expert financial analyst.
        TASK: Provide a detailed answer to the user's question using the retrieved documents.
        REQUIREMENTS:
        1. Write 200-300 words
        2. Use MARKDOWN formatting:
        - ## Headings
        - **Bold** for key metrics
        - Bullet points for lists
        - Tables for comparisons
        3. Include inline citations [1], [2], [3]
        4. Add a References section at the end with format: "Company: x, Year: y, Quarter: z, Page: n"
        Be thorough and detailed. Use all relevant information from the documents.
        Here is the User Question: {user_question}
        Here is the Retrived Documents:
        {retrieved_docs}
        Provide the detailed answer with citation:
        """
    response = llm.invoke(prompt)
    return {'messages': [response]} # [HumanMessage, AIMessage]

def should_rewrite(state:AgentState):
    is_relevant = state.get('is_relevant', True)
    if is_relevant:
        print(f"[ROUTER] Document is relevant -  procedding to answer")
        return "answer"
    else:
        print(f"[ROUTER] Documents are not relevant - rewritting the user query")
        return 'rewrite'

In [ ]:
def create_crag_agent():
    builder = StateGraph(AgentState)
    # add nodes
    builder.add_node('retriever', retrieve_node)
    builder.add_node('grade', grade_node)
    builder.add_node('rewrite', rewrite_query_node)
    builder.add_node('web_search', web_search_node)
    builder.add_node('answer', answer_node)
    # define edge
    builder.add_edge(START, 'retriever')
    builder.add_edge('retriever', 'grade')
    builder.add_conditional_edges('grade', should_rewrite, ['rewrite', 'answer'])
    builder.add_edge('rewrite', 'web_search')
    builder.add_edge('web_search', 'answer')
    builder.add_edge('answer', END)
    graph = builder.compile()
    return graph

agent = create_crag_agent()
agent

In [ ]:
query = "what is amazon's revenue in 2023?"
result = agent.invoke({'messages': [HumanMessage(query)]})
result['messages'][-1].pretty_print()

query = "what is amazon's and goodles revenue in 2023?"
result = agent.invoke({'messages': [HumanMessage(query)]})
result['messages'][-1].pretty_print()

query = "what is Tesla revenue in 2023?"
result = agent.invoke({'messages': [HumanMessage(query)]})
result['messages'][-1].pretty_print()